#

In [1]:
from pyIMSRG import *
import numpy as np

emax =3        # maximum number of oscillator quanta in the model space
ref = 'He4'     # reference used for normal ordering
val = ref # valence space

core_generator = 'atan'   # definition of generator eta for decoupling the core (could also use 'white')
smax_core = 10      # limit of integration in flow parameter s for first stage of decoupling
#smax_core = 0       # limit of integration in flow parameter s for first stage of decoupling

##### Example format of how to read input interaction matrix elements from file (these are not included with the code)
#f2b='input/TwBME-HO_NN-only_N3LO_EM500_srg1.8_hw16_emax14_e2max28.me2j.gz'
#f2e1,f2e2,f2l = 14,28,14
#f3b='input/NO2B_ThBME_EM1.8_2.0_3NFJmax15_IS_hw16_ms18_36_18.stream.bin'
#f3e1,f3e2,f3e3 = 18,36,18
#mode3n='no2b'
#LECs = 'EM1820'
#hw=16

#### Example format of how to read input interaction matrix elements from file (these are not included with the code)
f2b='input/TwBME-HO_NN-only_N3LO_EM500_srg1.8_hw16_emax14_e2max28.me2j.gz'
f2e1,f2e2,f2l = 14,28,14
f3b='input/NO2B_ThBME_EM7.5_1.8_2.0_IS_hw16from16_ms14_28_18.me3j.gz'
f3e1,f3e2,f3e3 = 14,28,18
mode3n='no2b'
LECs = 'EM7.5_1820'
hw=16



#hw = 20    # harmonic oscillator basis frequency
#LECs='Minnesota'
#f3b='none'

##########################################################################
###  END PARAMETER SETTING. BEGIN ACTUALLY DOING STUFF ##################
##########################################################################


### Create an instance of the ModelSpace class
ms = ModelSpace(emax,ref,val)
ms.SetHbarOmega(hw)

### the ReadWrite object handles reading and writing of files
rw = ReadWrite()

rank_j, parity, rank_Tz, particle_rank = 0,0,0,2
if f3b != 'none':
   particle_rank = 3

### Create an instance of the Operator class, representing the Hamiltonian
H = Operator(ms,rank_j, parity, rank_Tz, particle_rank)

### Either generate the matrix elements of the Minnesota potential, or read in matrix elements from file
if LECs == 'Minnesota':
    H += OperatorFromString(ms,'VMinnesota')

else:
  ### Read Two-body matrix elements
  rw.ReadBareTBME_Darmstadt(f2b,H,f2e1,f2e2,f2l)
  ### Read Three-body matrix elements
  if f3b != 'none':
     if mode3n == 'no2b':
        H.ThreeBody.SetMode('no2b')
        H.ThreeBody.ReadFile([f3b],[f3e1,f3e2,f3e3])
     else:
        rw.Read_Darmstadt_3body(f3b,H,f3e1,f3e2,f3e3)


### Add the relative kinetic energy, so H = Trel + V
H += OperatorFromString(ms,'Trel')
print('after reading files, 3-body norm is',H.ThreeBodyNorm())

### Create an instance of the HartreeFock class, used for solving the Hartree-Fock equations
hf = HartreeFock(H)
hf.Solve()
hf.PrintSPEandWF()

### Do normal ordering with respect to the HF basis, and retain only up to 2-body operators
HNO = hf.GetNormalOrderedH(2)

### Create an instance of the IMSRGSolver class, used for solving the IMSRG flow equations
imsrgsolver = IMSRGSolver(HNO)
imsrgsolver.SetMethod('magnus')  # Solve using the Magnus formulation. Could also be 'flow_RK4'

imsrgsolver.SetGenerator(core_generator)
imsrgsolver.SetSmax(smax_core)

### Do the first stage of integration to decouple the core
imsrgsolver.Solve()


### Hs is the IMSRG-evolved Hamiltonian
Hs = imsrgsolver.GetH_s()



Read 5696 matrix elements 
after reading files, 3-body norm is 130.51175702074318
ReadFile. from the input, I extracted input/NO2B_ThBME_EM7.5_1.8_2.0_IS_hw16from16_ms14_28_18.me3j.gz  14 28 18 14
ReadFile  reading/storing with 32  bit floats. filemode is gz
Allocated a vector of size 4312200
Done reading
Calculating moshinsky with Lmax = 3
done calculating moshinsky (389 elements)
Hash table has 397 buckets and a load factor 0.979849  estimated storage ~ 1.17123e-05 GB
HartreeFock::BuildMonopoleV3  storing 26624 doubles for Vmon3 and 26624 uint64's for Vmon3_keys.
HF converged after 23 iterations. 
e1hf = 43.2508884
e2hf = -65.6202531
e3hf = 3.1981722
EHF = -19.1711925
  i:   n   l  2j 2tz            SPE         occ.   |    overlaps
  0:   0   0   1  -1     -19.190740     1.000000   |  0.993783   0.111334  
  1:   0   0   1   1     -20.006811     1.000000   |  0.993278   0.115755  
  2:   0   1   3  -1       3.676799     0.000000   |  0.988166  -0.153387  
  3:   0   1   3   1       2

In [2]:
cm=Commutator
gm=Generator()

In [5]:
## initialize the T and D^dagger

def htc(Haml, chi):
    
    ## generate a antihermit chi
    chi_d = gm.GetEOM_ladder(chi, 1)
    
    ht_plus= chi*0
    ht_minus= chi*0
    
    ht_plus.SetAntiHermitian()
    
    ht_minus.SetHermitian()
    
    ht_plus = cm.Commutator(Haml, chi )
    ht_minus = cm.Commutator(Haml, chi_d )

    
    heom1= gm.GetEOM_ladder(ht_plus, 0)
    
    heom2= gm.GetEOM_ladder(ht_minus, 0)
    
    hod = (heom1+heom2)/2

    hod.SetHermitian()

    return(hod)




def Norm(T1, T2):
    return(gm.GetEOM_Overlap(T1,T2))

import numpy as np

def lanczos_proc( hv_func, norm_func, haml, vi, ndim):
    lanczos_vector = []
    hall = np.zeros([ndim,ndim])
    hall[0,0]=0.

    ## normalize it to 1
    nn=norm_func(vi,vi)
    print(nn)
    vi=vi/np.sqrt(nn)
    lanczos_vector.append(vi)

    for j in range(ndim):
        
        w = hv_func(haml,lanczos_vector[j])
    
        ai=norm_func(w,lanczos_vector[j])
        
    
        if(j>0):
            w=w-ai*lanczos_vector[j]-bj*lanczos_vector[j-1]
        else:
            w=w-ai*lanczos_vector[j]
        
        hall[j,j]=ai


        bj = np.sqrt(norm_func(w,w))
        #print(j,Norm(w,w), norm_func(w,w))
        #print(j, norm_func(w,w),ai,bj)
        if bj < 0.00001 :
            break
        lanczos_vector.append(w/bj)
    
        if(j<ndim-1):
            hall[j,j+1]=bj
            hall[j+1,j]=bj
        #print(j,ai,bj)
   # print(hall)
    e,v = np.linalg.eig(hall)
    
    return(e,v,lanczos_vector)


In [20]:
ndim=50
unt = UnitTest(ms)
rank_j, parity, rank_Tz, particle_rank, herm= 1,0,1,2,1
h3= unt.RandomOp( ms, rank_j,  rank_Tz, parity, particle_rank,herm)
#h3.MakeReduced()
chi= gm.GetEOM_ladder(h3,0)
cnorm=Norm(chi,chi)
chi=chi/np.sqrt(cnorm)
e,v,lvs = lanczos_proc( htc, Norm, Hs, chi, ndim)
e=np.sort(e)
print(e)

1.0000000000000002In  RandomOp  norm of 1b : 23.238   norm of 2b 1798.955

[ 23.3073914   27.09263178  28.89478924  30.64084204  31.78128011
  32.58482265  32.9635018   33.16442327  33.31588729  34.30985979
  39.5774379   41.24051767  41.84359405  42.83450537  43.37368686
  44.56117529  47.65850264  51.21484111  51.90678287  52.85716487
  57.57467815  58.33928758  59.98301385  62.47976977  63.61203651
  64.71320031  67.42835575  68.27178653  71.16617038  71.90803147
  76.88127918  77.27481717  80.80930514  81.43228334  83.20116502
  84.20770282  84.85467606  86.59740345  88.77999962  89.0594075
  92.05535438  92.19417595  95.43692825  95.47758921  95.55993095
  98.27989241  99.73968662  99.7692414   99.8829957  100.19483118]
